In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
train = pd.read_csv("/content/drive/MyDrive/IEEE_Comp/one-million-clicks-later/train.csv")
test = pd.read_csv("/content/drive/MyDrive/IEEE_Comp/one-million-clicks-later/test.csv")

In [ ]:
train

,user_id,video_id,video_duration,watch_time,liked,commented,subscribed_after,category,device,watch_time_of_day,recommended,clicked,timestamp,watch_percent,id
0,59445,40936,1134.886052,1287.412446,1,0,0,Sports,Tablet,Afternoon,1,0.0,2025-09-07 02:10:34,1.000000,283860
1,55829,17468,1335.223001,1224.760878,0,0,0,Gaming,Mobile,Night,1,0.0,2025-09-22 04:16:24,NaN,632997
2,68379,41436,2880.210321,1506.440934,1,0,0,Comedy,Desktop,Morning,0,0.0,2025-09-14 13:48:16,0.588424,94152
3,70789,17131,2975.577309,2327.012776,0,0,0,Comedy,TV,Evening,0,0.0,2024-01-21 03:33:17,NaN,483728
4,15748,1956,1022.594859,1041.854002,0,0,0,Gaming,Tablet,Evening,0,0.0,2023-02-03 19:40:06,0.978261,189031
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
610305,55018,27734,447.611469,426.135326,0,1,0,Comedy,Mobile,Night,1,0.0,2024-10-17 04:21:20,1.000000,573996
610306,76409,40910,2931.250360,1142.981047,0,0,0,Sports,Mobile,Morning,1,1.0,2024-08-02 19:31:31,0.370058,354793
610307,13481,38711,887.514810,979.867260,0,0,0,Music,Desktop,Night,1,0.0,2024-08-06 20:47:08,1.000000,659329
610308,19306,43932,2039.043448,659.272177,0,1,0,Music,TV,Morning,0,0.0,2024-09-03 15:10:18,0.367760,158349


In [ ]:
train.rename(columns={'video_duration': 'video_duration(sec)'}, inplace=True)
test.rename(columns={'video_duration': 'video_duration(sec)'}, inplace=True)

train.rename(columns={'watch_time': 'watch_time(sec)'}, inplace=True)
test.rename(columns={'watch_time': 'watch_time(sec)'}, inplace=True)

In [ ]:
train["clicked"].unique() ##target - 0,1,2
train["clicked"].value_counts() ## only 2 values are clicked = 2
train[train["clicked"] == 2]
train["clicked"] = train["clicked"].replace({2: 1})
train["clicked"].unique() #0 -not clicked,1 - clicked


array([0., 1.])

In [ ]:
train["liked"].unique() # 0, 1, 2, 'no', 'yes'
train["liked"].value_counts() # 0- 395602, 1- 169864, 2- 600, 'no'- 571, 'yes'- 548
train[train["liked"] == "2"]
train["liked"] = train["liked"].replace({"yes": "1", "no": "0", "2": "1"}) ####mapping 2 also as 1 since very less occurances of 2 compared to the size of datset
print("train liked uniques: \n",train["liked"].unique()) #

test["liked"].unique()
test["liked"].value_counts()
test[test["liked"] == "2"]
test["liked"] = test["liked"].replace({"yes": "1", "no": "0", "2": "1"}) ####mapping 2 also as 1 since very less occurances of 2 compared to the size of datset
print("test liked uniques: \n",test["liked"].unique())

liked_nan_train = train["liked"].isna().sum()
liked_nan_test = test["liked"].isna().sum()
print(f"total nan values in train liked: {liked_nan_train} out of {train.shape[0]} which is {liked_nan_train/train.shape[0]*100} %")
print(f"total nan values in test liked: {liked_nan_test} out of {test.shape[0]} which is {(liked_nan_test/test.shape[0]*100)} %")

## planning to fill the nan values with random 0's and 1's in same proportion as they are present in this feature


train liked uniques: 
 ['1' '0' nan]
test liked uniques: 
 ['0' '1' nan]
total nan values in train liked: 43125 out of 610310 which is 7.0660811718634795 %
total nan values in test liked: 10833 out of 152578 which is 7.099975094705659 %


In [ ]:
train['liked_missing'] = train['liked'].isnull().astype(int) #flag for the model so it understands that its a missing value
test['liked_missing'] = test['liked'].isnull().astype(int)

train['liked'] = pd.to_numeric(train['liked'], errors='coerce').fillna(0).astype(int)
test['liked'] = pd.to_numeric(test['liked'], errors='coerce').fillna(0).astype(int)

In [ ]:
train.groupby(["liked"])["clicked"].mean()

,clicked
liked,
0,0.146304
1,0.145890


In [ ]:
print("train 'commented' uniques",train["commented"].unique())
train["commented"].value_counts()

train["commented"].isna().sum()
test["commented"].isna().sum()   ## both test and train -- no nan values

train 'commented' uniques [0 1]


np.int64(0)

In [ ]:
train.groupby(["commented"])["clicked"].mean()

,clicked
commented,
0,0.146195
1,0.146128


## category encoding, liked nan values

In [ ]:
train["category"].unique()
train["category"] = train["category"].replace({'gamingg': 'Gaming', 'music': 'Music', 'comedy': 'Comedy', 'COMEDY': 'Comedy', 'Ed': 'Education','MUsic': 'Music'})
test["category"] = test["category"].replace({'gamingg': 'Gaming', 'music': 'Music', 'comedy': 'Comedy', 'COMEDY': 'Comedy', 'Ed': 'Education','MUsic': 'Music'})

train["category"].unique()
train["category"] = train["category"].apply(lambda x: x.strip())
test["category"] = test["category"].apply(lambda x: x.strip())

print(train.groupby(["category"], as_index = False)["clicked"].mean()) ## very less variations

print("value counts: \n", train["category"].value_counts())
train["commented"].isna().sum()
test["commented"].isna().sum() ## no nan values for both

## can use one-hot encoding but no hope, cant find any insightful info related to category for feature engineering, ahhhhhhhhhhhhhh!!!

    category   clicked
0     Comedy  0.145328
1  Education  0.146714
2     Gaming  0.146096
3  Lifestyle  0.144718
4      Music  0.148195
5       News  0.145210
6     Sports  0.147700
7       Tech  0.145506
value counts: 
 category
Music        77715
Comedy       76964
Tech         76574
Gaming       76566
Education    76298
News         75415
Lifestyle    75402
Sports       75376
Name: count, dtype: int64


np.int64(0)

In [ ]:
# train.info()
# train["watch_time_abs(sec)"] = train["watch_time(sec)"].apply(lambda x: abs(x)) # Calculate absolute values first
# train["watch_time_int"] = pd.qcut(train["watch_time_abs(sec)"], 4)  #4 types
# # print()
# train.groupby("watch_time_int", as_index = False)["clicked"].mean()

# # train["watch_time_abs(sec)"]

In [ ]:
# user_total_watchtime = train.groupby('user_id')['watch_time(sec)'].sum().reset_index()
# user_total_watchtime.columns = ['user_id', 'user_total_watchtime']

# user_cat_watchtime = train.groupby(['user_id', 'category'])['watch_time(sec)'].sum().reset_index()
# user_cat_watchtime.columns = ['user_id', 'category', 'user_cat_watchtime_sum']

# user_cat_affinity = user_cat_watchtime.merge(user_total_watchtime, on='user_id')
# user_cat_affinity['user_cat_aff'] = (
#     user_cat_affinity['user_cat_watchtime_sum'] / user_cat_affinity['user_total_watchtime']
# )

# df = train.merge(
#     user_cat_affinity[['user_id', 'category', 'user_cat_aff']],
#     on=['user_id', 'category'],
#     how='left'
# )

# print(df.head())

# df["user_cat_aff"].describe()
# df['affinity_bin'] = pd.cut(df['user_cat_aff'],
#                              bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0],
#                              labels=['0-10%', '10-20%', '20-30%', '30-40%', '40-50%', '50-100%'])

# affinity_ctr = df.groupby('affinity_bin')['clicked'].agg(['mean', 'count'])
# print(affinity_ctr)  ## no predictive pattern for user * category * watchtime

In [ ]:
print(train["device"].value_counts())
train["device"].unique()
train["device"].isna().sum()  #no null values

device_ctr = train.groupby('device')['clicked'].agg(['mean', 'count'])

cat_device = train.groupby(['category', 'device'])['clicked'].agg(['mean', 'count'])
cat_device.columns = ['click_rate', 'count']
cat_device = cat_device.sort_values('click_rate', ascending=False)
# print(cat_device_ctr.head(20))
print(cat_device)


device
Tablet     152767
Mobile     152746
Desktop    152504
TV         152293
Name: count, dtype: int64
                   click_rate  count
category  device                    
Sports    TV         0.150272  18939
Music     Tablet     0.149399  19632
          Desktop    0.149371  19234
News      TV         0.149164  18778
Gaming    TV         0.149094  19196
Education Tablet     0.148671  18921
Comedy    Tablet     0.148471  19290
Music     Mobile     0.147956  19418
Tech      Tablet     0.147872  19172
          TV         0.147657  18868
Education Mobile     0.147566  19083
Sports    Desktop    0.147519  18723
Education TV         0.146656  19065
Sports    Mobile     0.146618  18954
          Tablet     0.146375  18760
News      Tablet     0.146336  19052
Tech      Mobile     0.146209  19267
Music     TV         0.146055  19431
Lifestyle Desktop    0.145808  18737
Gaming    Desktop    0.145570  19166
Lifestyle Mobile     0.145371  18807
Gaming    Tablet     0.145340  18873
Lifesty

In [ ]:
np.sort(test["watch_time(sec)"].unique())

array([  -315.94833563,   -307.0737425 ,   -305.91273842, ...,
       128574.68289308, 129877.75995971, 131913.54425175])

In [ ]:
train['watch_percent_'] = train['watch_time(sec)'] / train['video_duration(sec)']  #any two of watch_percent, watch_time and video duration
test['watch_percent_'] = test['watch_time(sec)'] / test['video_duration(sec)']

In [ ]:
category_engagement = train.groupby('category').agg({'commented': 'mean','subscribed_after': 'mean', 'watch_time(sec)': ['mean', 'median', 'std'],'watch_percent': ['mean', 'median']}).reset_index()
category_engagement

category commented subscribed_after watch_time(sec)               \
                  mean             mean            mean       median   
0     Comedy  0.098994         0.049686     1298.792271  1067.017678   
1  Education  0.097761         0.049805     1315.497560  1074.847875   
2     Gaming  0.101481         0.051616     1286.475832  1068.624005   
3  Lifestyle  0.099639         0.049959     1211.838655  1058.961929   
4      Music  0.099119         0.048742     1419.300646  1073.318769   
5       News  0.100179         0.051329     1207.084791  1050.919423   
6     Sports  0.099939         0.050056     1211.506072  1059.265008   
7       Tech  0.100360         0.050565     1304.825603  1074.438200   

               watch_percent            
           std          mean    median  
0  2651.503104           inf  1.000000  
1  2912.618087           inf  0.990556  
2  2473.324331           inf  0.997213  
3   853.291906      0.749378  0.994077  
4  4131.193512           inf  1.000000  
5   855.795500      0.746482  0.988658  
6   853.431224      0.748752  0.991976  
7  2723.709197           inf  1.000000

In [ ]:
user_category_history = train.groupby(['user_id', 'category']).agg({'clicked': 'mean','watch_time(sec)': 'mean','video_id': 'count'}).reset_index()
user_category_history

,user_id,category,clicked,watch_time(sec),video_id
0,1,Lifestyle,0.000000,3134.000962,1
1,1,News,0.000000,572.574360,2
2,1,Sports,0.000000,1460.075792,1
3,2,Comedy,0.000000,1251.854813,2
4,2,Music,0.500000,1479.358936,2
...,...,...,...,...,...
427458,99999,Tech,0.333333,1267.278647,3
427459,100000,Gaming,0.000000,343.767927,1
427460,100000,News,0.000000,482.375337,1
427461,100000,Sports,0.333333,1183.909705,3


In [ ]:
train["watch_time(sec)"].unique()
np.sort(train["watch_time(sec)"].unique())[::] # ascending order

train[train["watch_time(sec)"] < 0]["clicked"].value_counts()  # 309 rows with negative watch time

,count
clicked,
0.0,254
1.0,55


In [ ]:
train["watch_time(sec)"].describe() # -ve watch time???

,watch_time(sec)
count,610310.000000
mean,1282.582449
std,2468.501280
min,-342.773810
25%,495.535339
50%,1066.165293
75%,1813.053282
max,134320.539232


In [ ]:
print(train["device"].unique()) # one hot encoding suits best
train.groupby('device')['clicked'].mean().reset_index()

# print(f"Infinite values: {np.isinf(train['watch_percent']).sum()}")

['Tablet' 'Mobile' 'Desktop' 'TV']


,device,clicked
0,Desktop,0.145111
1,Mobile,0.145555
2,TV,0.146908
3,Tablet,0.147178


In [ ]:
train["watch_time_of_day"].unique() # label encoding works but differene between night and morning should not be 3-0 = 0, it should be 1 since morning comes after night.  morning - 0, afternoon - 1, evening -2, night-3(degree wise kind off).
train.groupby('watch_time_of_day')['clicked'].mean().reset_index()


,watch_time_of_day,clicked
0,Afternoon,0.145520
1,Evening,0.146754
2,Morning,0.145794
3,Night,0.146685


In [ ]:
train["clicked"].unique()
train["clicked"].value_counts()

,count
clicked,
0.0,521090
1.0,89220


In [ ]:
# from datetime import datetime, timedelta

# train["timestamp"]  ## YYYY-MM-DD HH:MM:SS
# df = train.copy()
# df['timestamp'] = pd.to_datetime(df['timestamp'])
# print(f"\nTimestamp range:")
# print(f"  Start: {df['timestamp'].min()}")
# print(f"  End:   {df['timestamp'].max()}")
# print(f"  Span:  {(df['timestamp'].max() - df['timestamp'].min()).days} days")

# # Check for missing values
# missing_ts = df['timestamp'].isnull().sum()
# print(f"\nMissing timestamps: {missing_ts} ({missing_ts/len(df)*100:.2f}%)")

In [ ]:
import pandas as pd
import numpy as np
#missing timestampsk
train['timestamp_missing'] = train['timestamp'].isnull().astype(int)
train['timestamp'] = pd.to_datetime(train['timestamp'], errors='coerce')
train['hour'] = train['timestamp'].dt.hour
train['day'] = train['timestamp'].dt.dayofweek
train['hour'] = train['hour'].fillna(train['hour'].median())  #replace the nan with the median of hour(0-23)
train['day'] = train['day'].fillna(method='ffill')  #replace the nan day with next day


test['timestamp_missing'] = test['timestamp'].isnull().astype(int)
test['timestamp'] = pd.to_datetime(test['timestamp'], errors='coerce')
test['hour'] = test['timestamp'].dt.hour
test['day'] = test['timestamp'].dt.dayofweek
test['hour'] = test['hour'].fillna(test['hour'].median())  #replace the nan with the median of hour(0-23)
test['day'] = test['day'].fillna(method='ffill')  #replace the nan day with next day

/tmp/ipython-input-4038543177.py:9: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  train['day'] = train['day'].fillna(method='ffill')  #replace the nan day with next day
/tmp/ipython-input-4038543177.py:17: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  test['day'] = test['day'].fillna(method='ffill')  #replace the nan day with next day


In [ ]:
train['is_weekend'] = (train['day'] >= 5).astype(int)    ## if its a weekend
test['is_weekend'] = (test['day'] >= 5).astype(int)    ## if its a weekend

train.groupby(["is_weekend"])["clicked"].mean()

,clicked
is_weekend,
0,0.146275
1,0.145971


In [ ]:
train['is_prime_time'] = ((train['hour'] >= 18) & (train['hour'] <= 23)).astype(int) ## we can keep it prime time anything i picked 6pm to midnight
test['is_prime_time'] = ((test['hour'] >= 18) & (test['hour'] <= 23)).astype(int)

train.groupby(["is_prime_time"])["clicked"].mean()

,clicked
is_prime_time,
0,0.146377
1,0.145617


In [ ]:
user_device_counts = train.groupby(['user_id', 'device']).size().reset_index(name='device_count')
user_total_counts = train.groupby('user_id').size().reset_index(name='total_count')
user_device_pref_df = user_device_counts.merge(user_total_counts, on='user_id')
user_device_pref_df['user_device_pref'] = (user_device_pref_df['device_count'] / user_device_pref_df['total_count'])
train = train.merge(user_device_pref_df[['user_id', 'device', 'user_device_pref']],on=['user_id', 'device'], how='left')
 ##user's preference for the device from 0 to 1 \| user* device


user_device_counts = test.groupby(['user_id', 'device']).size().reset_index(name='device_count')
user_total_counts = test.groupby('user_id').size().reset_index(name='total_count')
user_device_pref_df = user_device_counts.merge(user_total_counts, on='user_id')
user_device_pref_df['user_device_pref'] = (user_device_pref_df['device_count'] / user_device_pref_df['total_count'])
test = test.merge(user_device_pref_df[['user_id', 'device', 'user_device_pref']],on=['user_id', 'device'], how='left')


In [ ]:
#one-hot encoding device and category
train = pd.get_dummies(train, columns=['device'], prefix='device', drop_first=False)
test = pd.get_dummies(test, columns=['device'], prefix='device', drop_first=False)

train = pd.get_dummies(train, columns=['category'], prefix='device', drop_first=False)
test = pd.get_dummies(test, columns=['category'], prefix='device', drop_first=False)


In [ ]:
print(train.shape)
print(test.shape)

(610310, 33)
(152578, 32)


In [ ]:
# is_weekend, is_prime_time, timestamp_missing
features = [
    'watch_time(sec)', 'watch_percent', 'liked', 'commented', 'subscribed_after','hour', 'day_of_week', 'is_weekend', 'is_prime_time','timestamp_missing','user_device_pref',
    ]

x_train = train.copy()
x_train = x_train.drop(columns = ["user_id", "video_id", "watch_time_of_day", "timestamp", "watch_percent","id","watch_percent_", "clicked"])
y_train = train["clicked"]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 610310 entries, 0 to 610309
Data columns (total 33 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   user_id              610310 non-null  int64         
 1   video_id             610310 non-null  int64         
 2   video_duration(sec)  610310 non-null  float64       
 3   watch_time(sec)      610310 non-null  float64       
 4   liked                610310 non-null  int64         
 5   commented            610310 non-null  int64         
 6   subscribed_after     610310 non-null  int64         
 7   watch_time_of_day    610310 non-null  object        
 8   recommended          610310 non-null  int64         
 9   clicked              610310 non-null  float64       
 10  timestamp            609104 non-null  datetime64[ns]
 11  watch_percent        556381 non-null  float64       
 12  id                   610310 non-null  int64         
 13  liked_missing 

In [ ]:
y_train = y_train.astype(int)
bool_cols = x_train.select_dtypes(bool).columns
x_train[bool_cols] = x_train[bool_cols].astype(int)

print(x_train.info())
print(y_train.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 610310 entries, 0 to 610309
Data columns (total 25 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   video_duration(sec)  610310 non-null  float64
 1   watch_time(sec)      610310 non-null  float64
 2   liked                610310 non-null  int64  
 3   commented            610310 non-null  int64  
 4   subscribed_after     610310 non-null  int64  
 5   recommended          610310 non-null  int64  
 6   liked_missing        610310 non-null  int64  
 7   timestamp_missing    610310 non-null  int64  
 8   hour                 610310 non-null  float64
 9   day                  610310 non-null  float64
 10  is_weekend           610310 non-null  int64  
 11  is_prime_time        610310 non-null  int64  
 12  user_device_pref     610310 non-null  float64
 13  device_Desktop       610310 non-null  int64  
 14  device_Mobile        610310 non-null  int64  
 15  device_TV        

In [ ]:
x_test = test.copy()
x_test = x_test.drop(columns = ["user_id", "video_id", "watch_time_of_day", "timestamp", "watch_percent","id","watch_percent_"])
x_train["watch_time(sec)"] =  x_train["watch_time(sec)"].apply(lambda x: abs(x))
x_test["watch_time(sec)"] =  x_test["watch_time(sec)"].apply(lambda x: abs(x))

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report


In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
rf_model.fit(x_train, y_train)
rf_preds = rf_model.predict(x_test)

In [ ]:
importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': x_train.columns,
    'importance': importances
}).sort_values('importance', ascending=False)

print(feature_importance_df.head(15))


# x_train_selected = x_train[[
#     "watch_time(sec)",
#     "video_duration(sec)",
#     "user_device_pref",
#     "hour",
#     "day",
#     "liked",
#     "recommended",
#     "commented",
#     "liked_missing",
#     "device_Mobile"
# ]]

# x_test_selected = x_test[[
#     "watch_time(sec)",
#     "video_duration(sec)",
#     "user_device_pref",
#     "hour",
#     "day",
#     "liked",
#     "recommended",
#     "commented",
#     "liked_missing",
#     "device_Mobile"
# ]]

# rf_model_ = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
# rf_model_.fit(x_train_selected, y_train)


x_train_new = train["watch_time(sec)", "video_duration(sec)", "user_device_pref", "hour", "commented", "liked", "recommended"]
x_test_new = test["watch_time(sec)", "video_duration(sec)", "user_device_pref", "hour", "commented", "liked", "recommended"]

                feature  importance
1       watch_time(sec)    0.265136
0   video_duration(sec)    0.264892
12     user_device_pref    0.148429
8                  hour    0.123497
9                   day    0.055137
2                 liked    0.016236
5           recommended    0.014039
3             commented    0.010793
6         liked_missing    0.008353
14        device_Mobile    0.007433
15            device_TV    0.006951
16        device_Tablet    0.006793
13       device_Desktop    0.006625
19        device_Gaming    0.006490
4      subscribed_after    0.006371


KeyError: ('watch_time(sec)', 'video_duration(sec)', 'user_device_pref', 'hour', 'commented', 'liked', 'recommended')

In [ ]:
"rf_preds_ = rf_model_.predict(x_test_selected)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)
xgb_model.fit(x_train, y_train)
xgb_preds = xgb_model.predict(x_test)

In [ ]:
lgb_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)
lgb_model.fit(x_train, y_train)
lgb_preds = lgb_model.predict(x_test)

[LightGBM] [Info] Number of positive: 89220, number of negative: 521090
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.180956 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 659
[LightGBM] [Info] Number of data points in the train set: 610310, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.146188 -> initscore=-1.764818
[LightGBM] [Info] Start training from score -1.764818


In [ ]:
sub_rf_ = pd.DataFrame({"id": test["id"],"clicked": rf_preds_})


# sub_lgb = pd.DataFrame({"id": test["id"],"clicked": lgb_preds})


sub_rf_.to_csv("/content/drive/MyDrive/IEEE_Comp/one-million-clicks-later/rf_new.csv", index=False)
# sub_lgb.to_csv("/content/drive/MyDrive/IEEE_Comp/one-million-clicks-later/lgb.csv", index=False)


In [ ]:
sub_xgb = pd.DataFrame({"id": test["id"],"clicked": xgb_preds})
sub_xgb.to_csv("/content/drive/MyDrive/IEEE_Comp/one-million-clicks-later/xgb.csv", index=False)

In [ ]:
print(sub_rf[sub_rf["clicked"] == 1]) ## no-ne

            id  clicked
1453     98820        1
3019    573675        1
3262    540929        1
4567    295407        1
5066    201438        1
...        ...      ...
148838  338760        1
148901  542684        1
148969  744009        1
150764   83821        1
151376  599346        1

[235 rows x 2 columns]


In [ ]:
train["clicked"].value_counts()

,count
clicked,
0.0,521090
1.0,89220
